# S2 Activation: Smoothness, Spectral Leakage, and Equivariance

This report systematically studies the chain:

$$\text{Activation Smoothness} \;\rightarrow\; \text{Spectral Leakage} \;\rightarrow\; \text{Equivariance Error} \;\rightarrow\; \text{Task Performance}$$

**S2 Activation** is the standard nonlinearity used in SO(3)-equivariant neural networks (SCN, eSCN, EquiformerV2, etc.). It works by:
1. Projecting spherical harmonic (SH) coefficients onto the sphere
2. Applying a pointwise nonlinearity
3. Projecting back to SH coefficients via quadrature

The nonlinearity generates high-frequency components beyond $l_{\max}$, which are lost upon truncation. With insufficient quadrature resolution, these components also **alias** into the low-$l$ coefficients, breaking equivariance.

---

## Experiments Overview

| Experiment | Question | Key Variable |
|:---|:---|:---|
| **A. Spectral Leakage** | How much energy does each activation spread beyond $l_{\max}$? | Activation function |
| **B. Coefficient Error** | How does sampling resolution affect S2Activation output accuracy? | Activation $\times$ Sampling |
| **C. Equivariance Error** | Does S2Act(D$\cdot$x) = D$\cdot$S2Act(x)? | Activation $\times$ Sampling |
| **D. Task Performance** | Do spectral/equivariance findings predict downstream accuracy? | Activation $\times$ Sampling |

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from IPython.display import Image, display, Markdown
import pandas as pd

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

# Load all results
with open('results/metrics/expA_spectral.json') as f:
    expA = json.load(f)
with open('results/metrics/expB_coefficient_error.json') as f:
    expB = json.load(f)
with open('results/metrics/expC_equivariance.json') as f:
    expC = json.load(f)
with open('results/metrics/expD_task.json') as f:
    expD = json.load(f)

print('All results loaded successfully.')

---
## Experiment A: Spectral Leakage Analysis

**Goal**: Measure how different nonlinear activation functions spread energy into spherical harmonic components beyond $l_{\max}$.

**Method**: 
- Generate random band-limited input ($l \le l_{\max}$)
- Apply nonlinearity pointwise on the sphere
- Project back using high-resolution GL quadrature ($l_{\max,\text{ref}} = 25$)
- Compute power spectrum $P(l) = \sum_m |c_{lm}|^2$ and leakage ratio $R = \sum_{l > l_{\max}} P(l) / \sum_l P(l)$

**Activations tested**: ReLU, abs, LeakyReLU, SiLU, GELU, tanh, Softplus($\beta$) for $\beta \in \{1, 3, 10, 30, 100\}$, $x^2$, sin

In [ ]:
# Table A1: Leakage ratios for all activations
rows = []
for lmax in ['3', '6', '10']:
    for act_name, act_data in expA[lmax].items():
        rows.append({
            'l_max': int(lmax),
            'Activation': act_name,
            'Leakage Ratio R': act_data['leakage_ratio'],
            'R std': act_data['leakage_ratio_std'],
            'l_effective': act_data['l_effective'],
        })

df_A = pd.DataFrame(rows)
# Pivot for cleaner view
pivot_A = df_A.pivot_table(index='Activation', columns='l_max', values='Leakage Ratio R')
pivot_A = pivot_A.sort_values(by=3)  # Sort by l_max=3 leakage

print('Table A1: Leakage Ratio R (fraction of energy above l_max)')
print('=' * 60)
display(pivot_A.style.format('{:.4f}').background_gradient(cmap='RdYlGn_r', axis=None))

### A1: Power Spectra After Nonlinearity

The plot below shows $P(l)$ for selected activations. The vertical dashed line marks $l_{\max}$. Energy to the right is "leaked" spectral content.

In [ ]:
display(Image('results/figures/expA_power_spectra.png', width=900))

**Observations**:
- **$x^2$** has exact finite bandwidth at $2 l_{\max}$ (product of two band-limited signals)
- **ReLU / abs** (C$^0$) have broad power-law spectral tails
- **SiLU / GELU** (C$^\infty$) have faster spectral decay
- **Softplus($\beta$)** interpolates continuously between smooth (small $\beta$) and sharp (large $\beta$)

### A2: Softplus Family - Continuous Smoothness Control

In [ ]:
display(Image('results/figures/expA_softplus_spectra.png', width=900))

In [ ]:
display(Image('results/figures/expA_softplus_transition.png', width=600))

**Key finding**: Softplus($\beta$) provides a one-parameter family that smoothly transitions from near-linear (small $\beta$, minimal leakage) to ReLU-like (large $\beta$, maximal leakage). This confirms the theoretical prediction that smoothness controls spectral leakage.

### A3: Leakage Ratio Comparison

In [ ]:
display(Image('results/figures/expA_leakage_ratios.png', width=900))

---
## Experiment B: S2Activation Coefficient Error

**Goal**: Measure how the full S2Activation pipeline's output coefficient error depends on (activation $\times$ sampling method $\times$ resolution).

**Method**:
- Ground truth: S2Activation with GL quadrature at $l_{\max,\text{ref}} = 25$, truncated to $l_{\max}$
- Test: S2Activation at various sampling configs
- Error = $\|\mathbf{c}_{\text{out}} - \mathbf{c}_{\text{gt}}\| / \|\mathbf{c}_{\text{gt}}\|$

**Sampling configs**: GL$\times$1/2/3, Lebedev min/2$\times$, Uniform 20/50/100

In [ ]:
# Table B1: Coefficient error summary
rows = []
for lmax in ['3', '6', '10']:
    for act_name, act_data in expB[lmax].items():
        for cfg_name, cfg_data in act_data.items():
            rows.append({
                'l_max': int(lmax),
                'Activation': act_name,
                'Sampling': cfg_name,
                'N_points': cfg_data['n_points'],
                'Rel Error': cfg_data['mean_rel_error'],
                'Trunc Ratio': cfg_data['mean_trunc_ratio'],
            })

df_B = pd.DataFrame(rows)

# Show GL results for l_max=6
mask = (df_B['l_max'] == 6) & df_B['Sampling'].str.startswith('GL')
pivot_B = df_B[mask].pivot_table(index='Activation', columns='Sampling', values='Rel Error')
pivot_B = pivot_B[['GL_1x', 'GL_2x', 'GL_3x']]
pivot_B = pivot_B.sort_values('GL_1x')

print('Table B1: Relative Coefficient Error (l_max=6, GL quadrature)')
print('=' * 60)
display(pivot_B.style.format('{:.3f}').background_gradient(cmap='RdYlGn_r', axis=None))

In [ ]:
# Show truncation ratios - should be constant across sampling methods
mask = (df_B['l_max'] == 6) & df_B['Sampling'].str.startswith('GL')
pivot_trunc = df_B[mask].pivot_table(index='Activation', columns='Sampling', values='Trunc Ratio')
pivot_trunc = pivot_trunc[['GL_1x', 'GL_2x', 'GL_3x']]

print('Table B2: Truncation Ratio (energy above l_max / total, l_max=6)')
print('Truncation is a property of the activation, NOT the sampling.')
print('=' * 60)
display(pivot_trunc.style.format('{:.4f}').background_gradient(cmap='RdYlGn_r', axis=None))

### B1: Error vs Number of Sampling Points

In [ ]:
display(Image('results/figures/expB_error_vs_npoints.png', width=900))

**Critical finding**: The coefficient error barely decreases with more sampling points. The error is **dominated by truncation** (losing $l > l_{\max}$ components), not by aliasing. Even 20,000 uniform points give almost the same error as 32 GL points.

This means:
- Oversampling provides diminishing returns for coefficient accuracy
- The only way to reduce S2Activation output error is to **use smoother activations** (less spectral leakage)

### B2: Error Heatmaps (Activation x Sampling)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, lmax in enumerate([3, 6, 10]):
    ax = axes[i]
    img = plt.imread(f'results/figures/expB_heatmap_lmax{lmax}.png')
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'l_max = {lmax}')
plt.tight_layout()
plt.show()

### B3: Error Decay with Oversampling Ratio

In [ ]:
display(Image('results/figures/expB_oversampling_decay.png', width=900))

---
## Experiment C: Equivariance Error

**Goal**: Directly measure how well S2Activation commutes with SO(3) rotations.

**Method**:
Equivariance requires $\text{S2Act}(D \cdot x) = D \cdot \text{S2Act}(x)$ where $D$ is the Wigner-D rotation matrix.

$$\epsilon_{\text{equiv}} = \frac{\|\text{S2Act}(D \cdot x) - D \cdot \text{S2Act}(x)\|}{\|\text{S2Act}(D \cdot x)\|}$$

Averaged over 20 random rotations $\times$ 10 random inputs.

In [ ]:
# Table C1: Equivariance error summary
rows = []
for lmax in ['3', '6', '10']:
    for act_name, act_data in expC[lmax].items():
        for cfg_name, cfg_data in act_data.items():
            rows.append({
                'l_max': int(lmax),
                'Activation': act_name,
                'Sampling': cfg_name,
                'N_points': cfg_data['n_points'],
                'Equiv Error': cfg_data['mean_equiv_error'],
                'Equiv Std': cfg_data['std_equiv_error'],
            })

df_C = pd.DataFrame(rows)

# Pivot: GL results for l_max=6
mask = (df_C['l_max'] == 6) & df_C['Sampling'].str.startswith('GL')
pivot_C = df_C[mask].pivot_table(index='Activation', columns='Sampling', values='Equiv Error')
pivot_C = pivot_C[['GL_1x', 'GL_2x', 'GL_3x']]
pivot_C = pivot_C.sort_values('GL_1x')

print('Table C1: Equivariance Error (l_max=6, GL quadrature)')
print('=' * 60)
display(pivot_C.style.format('{:.2e}').background_gradient(cmap='RdYlGn_r', axis=None))

### C1: Equivariance Error by Activation Function

In [ ]:
display(Image('results/figures/expC_equiv_vs_activation.png', width=900))

### C2: Equivariance Error vs Oversampling Ratio

Unlike coefficient error (Exp B), equivariance error **does** decrease significantly with oversampling.

In [ ]:
display(Image('results/figures/expC_equiv_vs_oversampling.png', width=900))

**Key finding**: While coefficient error is dominated by truncation (Exp B), equivariance error is strongly affected by aliasing and responds dramatically to oversampling:

| Config | ReLU | SiLU | Softplus(1) |
|:---|:---:|:---:|:---:|
| GL$\times$1 | 0.24 | 0.26 | 0.14 |
| GL$\times$2 | 0.03 | 0.01 | 0.004 |
| GL$\times$3 | 0.01 | 0.002 | 0.0003 |

Softplus(1) + GL$\times$3 achieves **800$\times$ better equivariance** than ReLU + GL$\times$1.

### C3: Equivariance Error vs Spectral Leakage

This plot validates the causal chain: more leakage $\rightarrow$ more equivariance error.

In [ ]:
display(Image('results/figures/expC_equiv_vs_leakage.png', width=900))

---
## Experiment D: Task Performance

**Goal**: Verify whether spectral/equivariance analysis predicts downstream task performance.

**Method**: Train a 1-layer SphericalCNN on a synthetic classification task where class-discriminative information resides primarily in high-$l$ coefficients. Different (activation $\times$ sampling) configurations are compared.

**Task**: 5-class classification, $l_{\max} = 6$, class signal concentrated at $l \geq 3$.

In [ ]:
# Table D1: Task performance summary
rows = []
for key, data in expD.items():
    rows.append({
        'Config': key,
        'Activation': data['activation'],
        'Sampling': data['sampling'],
        'Test Accuracy': data['mean_test_acc'],
        'Std': data['std_test_acc'],
        'Train Time (s)': data['mean_train_time'],
    })

df_D = pd.DataFrame(rows)
pivot_D = df_D.pivot_table(index='Activation', columns='Sampling', values='Test Accuracy')
pivot_D = pivot_D.sort_values(pivot_D.columns[0], ascending=False)

print('Table D1: Test Accuracy by (Activation x Sampling)')
print('=' * 60)
display(pivot_D.style.format('{:.3f}').background_gradient(cmap='RdYlGn', axis=None))

In [ ]:
display(Image('results/figures/expD_accuracy_comparison.png', width=700))

In [ ]:
display(Image('results/figures/expD_acc_vs_equiv.png', width=600))

**Note**: The accuracy differences are small (~1-2%) because this is a 1-layer model that can partially compensate for S2Activation errors via learned weights. In deeper networks where equivariance errors compound across layers, the differences would be more pronounced.

---
## Cross-Experiment Analysis: The Complete Chain

We now combine findings from all four experiments to validate the causal chain.

In [ ]:
# Build combined table: leakage (A) -> coefficient error (B) -> equivariance (C) -> accuracy (D)
# All at l_max=6, GL_1x sampling
lmax = '6'
combined_rows = []

common_acts = ['ReLU', 'abs', 'SiLU', 'GELU', 'tanh', 'Softplus_1', 'Softplus_10', 'Softplus_100']

for act in common_acts:
    row = {'Activation': act}
    
    # Exp A: leakage
    if act in expA.get(lmax, {}):
        row['Leakage R'] = expA[lmax][act]['leakage_ratio']
    
    # Exp B: coefficient error at GL_1x
    if act in expB.get(lmax, {}) and 'GL_1x' in expB[lmax][act]:
        row['Coeff Error'] = expB[lmax][act]['GL_1x']['mean_rel_error']
    
    # Exp C: equivariance at GL_1x
    if act in expC.get(lmax, {}) and 'GL_1x' in expC[lmax][act]:
        row['Equiv Error'] = expC[lmax][act]['GL_1x']['mean_equiv_error']
    
    # Exp D: task accuracy
    key = f"{act}_GL_1x"
    if key in expD:
        row['Test Acc'] = expD[key]['mean_test_acc']
    
    combined_rows.append(row)

df_combined = pd.DataFrame(combined_rows).set_index('Activation')

print('Combined Chain Analysis (l_max=6, GL_1x sampling)')
print('Smoothness -> Leakage -> Coeff Error -> Equiv Error -> Performance')
print('=' * 80)
display(df_combined.style.format({
    'Leakage R': '{:.4f}',
    'Coeff Error': '{:.3f}',
    'Equiv Error': '{:.4f}',
    'Test Acc': '{:.3f}',
}).background_gradient(subset=['Leakage R', 'Coeff Error', 'Equiv Error'], cmap='RdYlGn_r', axis=0
).background_gradient(subset=['Test Acc'], cmap='RdYlGn', axis=0))

In [ ]:
# Scatter: leakage vs equivariance error (with both GL_1x and GL_3x)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, cfg in enumerate(['GL_1x', 'GL_3x']):
    ax = axes[ax_idx]
    xs, ys, labels = [], [], []
    
    for act in common_acts:
        if act in expA.get(lmax, {}) and act in expC.get(lmax, {}):
            if cfg in expC[lmax].get(act, {}):
                R = expA[lmax][act]['leakage_ratio']
                E = expC[lmax][act][cfg]['mean_equiv_error']
                xs.append(R)
                ys.append(E)
                labels.append(act)
    
    ax.scatter(xs, ys, s=80, c='steelblue', edgecolors='navy', zorder=3)
    for i, label in enumerate(labels):
        ax.annotate(label, (xs[i], ys[i]), fontsize=9, xytext=(5, 5),
                   textcoords='offset points')
    
    # Fit trend line
    if len(xs) >= 3:
        from scipy.stats import pearsonr
        r, p = pearsonr(xs, ys)
        z = np.polyfit(xs, ys, 1)
        x_line = np.linspace(min(xs)*0.9, max(xs)*1.1, 50)
        ax.plot(x_line, np.polyval(z, x_line), '--', color='gray', alpha=0.5)
        ax.set_title(f'{cfg}: Pearson r = {r:.3f} (p = {p:.4f})')
    
    ax.set_xlabel('Leakage Ratio R (Exp A)')
    ax.set_ylabel('Equivariance Error (Exp C)')
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Spectral Leakage vs Equivariance Error (l_max={lmax})', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Oversampling benefit: how much does GL_3x improve over GL_1x?
rows = []
for act in common_acts:
    if act in expC.get(lmax, {}):
        act_data = expC[lmax][act]
        if 'GL_1x' in act_data and 'GL_3x' in act_data:
            e1 = act_data['GL_1x']['mean_equiv_error']
            e3 = act_data['GL_3x']['mean_equiv_error']
            rows.append({
                'Activation': act,
                'GL_1x': e1,
                'GL_3x': e3,
                'Improvement': e1 / e3 if e3 > 0 else float('inf'),
            })

df_improve = pd.DataFrame(rows).set_index('Activation')
df_improve = df_improve.sort_values('Improvement', ascending=False)

print(f'Table: Equivariance improvement from GL_1x to GL_3x (l_max={lmax})')
print('=' * 60)
display(df_improve.style.format({
    'GL_1x': '{:.4f}',
    'GL_3x': '{:.4f}',
    'Improvement': '{:.0f}x',
}).background_gradient(subset=['Improvement'], cmap='Greens'))

---
## Key Conclusions

### 1. Activation smoothness controls spectral leakage (Exp A)
- **C$^\infty$ activations** (SiLU, Softplus(1)) produce fast spectral decay, with leakage ratios $R \sim$ 3-10%
- **C$^0$ activations** (ReLU, abs) produce power-law tails, with $R \sim$ 9-16%
- **Softplus($\beta$)** provides a continuous 1-parameter family: $\beta \to \infty$ approaches ReLU

### 2. Coefficient error is dominated by truncation, not aliasing (Exp B)
- Increasing sampling from 32 to 20,000 points barely changes coefficient error
- The irreducible error floor comes from **truncating** high-$l$ content generated by the nonlinearity
- Smoother activations have lower truncation error because they generate less high-$l$ content

### 3. Equivariance error responds strongly to both smoothness AND sampling (Exp C)
- Unlike coefficient error, equivariance error **does** benefit dramatically from oversampling
- GL$\times$3 reduces equivariance error by 10-450$\times$ compared to GL$\times$1
- Smooth activations benefit **more** from oversampling (Softplus(1): 450$\times$ improvement vs ReLU: 23$\times$)
- **Best combo**: Softplus(1) + GL$\times$3 achieves $\epsilon_{\text{equiv}} \sim 3 \times 10^{-4}$

### 4. The truncation-aliasing dichotomy
- **Coefficient error** $\approx$ truncation error (sampling doesn't help much)
- **Equivariance error** $\approx$ aliasing error (sampling helps a lot)
- This explains why eSEN abandoned S2 Activation: even with good quadrature, the coefficients are inaccurate (truncation), and equivariance is only approximate (aliasing)

### 5. Practical recommendations
| Objective | Recommendation |
|:---|:---|
| Minimize equivariance error | Softplus(1) or SiLU + $\geq$2$\times$ oversampling |
| Minimize computational cost | SiLU + GL$\times$1 (already used by most models) |
| Parametric smoothness control | Softplus($\beta$) as tunable knob |
| Avoid | ReLU (high leakage, poor equivariance at any sampling) |

---

## Appendix: Phase 1 Experiment Results (Quadrature Method Comparison)

The earlier phase compared quadrature methods (GL, Lebedev, Uniform, Fibonacci) for SH reconstruction accuracy without nonlinearities.

In [ ]:
display(Markdown('### Reconstruction Accuracy (Exp 1)'))
display(Image('results/figures/exp1_accuracy_curves.png', width=900))

In [ ]:
display(Markdown('### Per-Degree Error (Exp 1)'))
display(Image('results/figures/exp1_error_by_degree.png', width=900))

In [ ]:
display(Markdown('### Computational Cost Scaling (Exp 2)'))
display(Image('results/figures/exp2_time_scaling.png', width=700))

In [ ]:
display(Markdown('### Efficiency Frontier (Exp 4)'))
display(Image('results/figures/exp4_efficiency_frontier.png', width=900))

In [ ]:
display(Markdown('### Asymptotic Complexity (Exp 4)'))
display(Image('results/figures/exp4_asymptotic.png', width=600))